# MediDiary — convert the fine-tuned Whisper (Hugging Face) → whisper.cpp ggml

The app runs Whisper through **whisper.cpp v1.9.5**, which cannot read Hugging Face `safetensors`.
This notebook converts your fine-tune with the converter **from the same whisper.cpp version the app is built with**,
optionally quantises it, transcribes a test clip with the converted file, and saves `medidiary-whisper.bin`.

Works with: a full fine-tune folder **or** a LoRA/PEFT adapter (merged automatically), on the Hub or in Google Drive.
Runtime: any Colab runtime (CPU is fine). ≈ 10 min.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
FINETUNED   = "/content/drive/MyDrive/your-whisper-finetune"   # Drive folder OR Hub id like "you/whisper-small-bn"
BASE_MODEL  = "openai/whisper-small"   # the model you fine-tuned FROM (used for LoRA merge + missing tokenizer files)
QUANT       = "q5_0"                   # None = keep f16. q5_0 ≈ 1/3 the size, little accuracy loss; q8_0 = safer
LANGUAGE    = "bn"                     # same as the app default (Settings → Speech language)
TEST_AUDIO  = None                     # path to a short Bangla .wav/.mp3/.m4a for the smoke test (None = upload prompt)
DRIVE_OUT   = "/content/drive/MyDrive/medidiary_whisper"
WHISPER_CPP_TAG = "v1.9.5"             # MUST match third_party/whisper.cpp in the app repo

import os
if FINETUNED.startswith("/content/drive") or DRIVE_OUT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
%%capture
!pip install -q "transformers>=4.40" peft safetensors sentencepiece
!git clone -q --depth 1 https://github.com/openai/whisper /content/openai-whisper
!git clone -q --depth 1 --branch {WHISPER_CPP_TAG} https://github.com/ggml-org/whisper.cpp /content/whisper.cpp

In [ ]:
# ---- Materialise a plain HF model folder (merge LoRA, add tokenizer files the converter needs) ----
import json, os, shutil, torch
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import WhisperForConditionalGeneration, WhisperTokenizer

src = Path(FINETUNED) if os.path.isdir(FINETUNED) else Path(snapshot_download(FINETUNED))
MODEL_DIR = Path("/content/hf_model"); shutil.rmtree(MODEL_DIR, ignore_errors = True); MODEL_DIR.mkdir()

if (src / "adapter_config.json").exists():
    from peft import PeftModel
    print("LoRA adapter detected → merging into", BASE_MODEL)
    base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype = torch.float32)
    merged = PeftModel.from_pretrained(base, str(src)).merge_and_unload()
    merged.save_pretrained(MODEL_DIR, safe_serialization = True)
else:
    WhisperForConditionalGeneration.from_pretrained(src, torch_dtype = torch.float32).save_pretrained(MODEL_DIR)

# convert-h5-to-ggml.py needs vocab.json AND added_tokens.json next to config.json.
try:
    tok = WhisperTokenizer.from_pretrained(src)
except Exception:
    print("No tokenizer in the fine-tune folder → using", BASE_MODEL)
    tok = WhisperTokenizer.from_pretrained(BASE_MODEL)
tok.save_pretrained(MODEL_DIR)
if not (MODEL_DIR / "added_tokens.json").exists():
    (MODEL_DIR / "added_tokens.json").write_text(json.dumps(tok.get_added_vocab(), ensure_ascii = False))

cfg = json.loads((MODEL_DIR / "config.json").read_text())
print("files:", sorted(p.name for p in MODEL_DIR.iterdir()))
print(f"d_model={cfg['d_model']} layers={cfg['encoder_layers']}/{cfg['decoder_layers']} mels={cfg['num_mel_bins']} vocab={cfg['vocab_size']}")

In [ ]:
# ---- Convert to ggml (f16) ----------------------------------------------------------------
OUT_DIR = Path("/content/ggml_out"); OUT_DIR.mkdir(exist_ok = True)
!python /content/whisper.cpp/models/convert-h5-to-ggml.py {MODEL_DIR} /content/openai-whisper {OUT_DIR}
F16 = OUT_DIR / "ggml-model.bin"
assert F16.exists(), "conversion failed — see the log above"
with open(F16, "rb") as f: magic = f.read(4)
assert magic == b"lmgg", f"unexpected magic {magic!r}"   # same header check the app does on import
print(f"{F16}  {F16.stat().st_size/1e6:.0f} MB")

In [ ]:
# ---- Build whisper.cpp tools (CPU) — ~3 min --------------------------------------------------
!cmake -S /content/whisper.cpp -B /content/whisper.cpp/build -DCMAKE_BUILD_TYPE=Release -DWHISPER_BUILD_TESTS=OFF > /dev/null
!cmake --build /content/whisper.cpp/build -j --target whisper-cli whisper-quantize 2>&1 | tail -2
BIN = "/content/whisper.cpp/build/bin"

In [ ]:
FINAL = F16
if QUANT:
    FINAL = OUT_DIR / f"ggml-model-{QUANT}.bin"
    !{BIN}/whisper-quantize {F16} {FINAL} {QUANT} 2>&1 | tail -3
print(f"final model: {FINAL}  {FINAL.stat().st_size/1e6:.0f} MB")

In [ ]:
# ---- Smoke test: transcribe with the CONVERTED file (same engine as the app) ------------------
import subprocess
if TEST_AUDIO is None:
    from google.colab import files
    print("Upload a short Bangla voice note (any format):")
    TEST_AUDIO = list(files.upload().keys())[0]
!ffmpeg -loglevel error -y -i "{TEST_AUDIO}" -ar 16000 -ac 1 -c:a pcm_s16le /content/test16k.wav   # app records 16 kHz mono PCM

for name, path in [("converted f16", F16)] + ([(f"quantised {QUANT}", FINAL)] if QUANT else []):
    r = subprocess.run([f"{BIN}/whisper-cli", "-m", str(path), "-f", "/content/test16k.wav", "-l", LANGUAGE, "-nt"],
                       capture_output = True, text = True)
    print(f"--- {name} ---\n{r.stdout.strip() or r.stderr[-1500:]}\n")
print("Compare with what your HF model gives on the same clip; f16 should be near-identical.")

In [ ]:
# ---- Save under the file name the app expects ---------------------------------------------
APP_NAME = "medidiary-whisper.bin"
shutil.copy(FINAL, f"/content/{APP_NAME}")
if DRIVE_OUT:
    os.makedirs(DRIVE_OUT, exist_ok = True)
    shutil.copy(FINAL, f"{DRIVE_OUT}/{APP_NAME}")
    print("saved:", f"{DRIVE_OUT}/{APP_NAME}")

### Put it on the phone
```bash
adb push medidiary-whisper.bin /sdcard/Android/data/com.medidiary.app/files/models/medidiary-whisper.bin
```
then **Settings → On-device AI models → Re-scan** (or **Import file…**). Set **Speech language** to the language you fine-tuned on.

Size guide (f16 → q5_0): tiny 75 → 30 MB · base 145 → 60 MB · small 485 → 190 MB · medium 1.5 GB → 540 MB.
On low-RAM phones prefer `small` or smaller; Gemma (~0.8 GB) is never loaded at the same time.